# 04 — Model Training (VaxFlow)

1. **Demand Forecasting** — SMA vs Exponential Smoothing (จูน α)  
2. **Transportation Model** — Linear Programming หาแผนโอนย้ายต้นทุนต่ำสุด (Proposal §4.3)

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
from scipy.optimize import linprog

ROOT = Path.cwd().parent if Path.cwd().name == 'notebook' else Path.cwd()
VAX = ROOT / 'data' / 'vaccine'
OUT = VAX / 'outputs'; OUT.mkdir(parents=True, exist_ok=True)
feat = pd.read_csv(VAX / 'features' / 'demand_features.csv', parse_dates=['date'])

## 1) Demand Forecasting — จูน α ของ Exponential Smoothing

แบ่ง train/test ตามเวลา (80/20) แล้ววัด RMSE ของ SMA-7 เทียบกับ ES ที่ α ต่าง ๆ

In [ ]:
def rmse(a, b):
    return float(np.sqrt(np.mean((np.asarray(a) - np.asarray(b)) ** 2)))

def es_forecast(series, alpha):
    f = series.iloc[0]
    out = []
    for t in range(len(series)):
        out.append(f)
        f = alpha * series.iloc[t] + (1 - alpha) * f
    return np.array(out)

alphas = [0.1, 0.2, 0.3, 0.4, 0.5, 0.7]
results = []
for (hid, pid), g in feat.groupby(['hospital_id', 'product_id']):
    g = g.sort_values('date'); cut = int(len(g) * 0.8)
    test = g.iloc[cut:]
    sma_rmse = rmse(test['demand'], test['sma_7'])
    best_a, best_r = None, np.inf
    for a in alphas:
        pred = es_forecast(g['demand'], a)[cut:]
        r = rmse(test['demand'], pred)
        if r < best_r: best_a, best_r = a, r
    results.append({'hospital_id': hid, 'product_id': pid,
                    'sma7_rmse': round(sma_rmse, 2), 'best_alpha': best_a,
                    'es_rmse': round(best_r, 2),
                    'winner': 'ES' if best_r < sma_rmse else 'SMA'})
fc = pd.DataFrame(results)
print(fc['winner'].value_counts().to_dict())
fc

In [ ]:
fc.to_csv(OUT / 'forecast_model_selection.csv', index=False, encoding='utf-8-sig')
print('mean SMA RMSE:', round(fc['sma7_rmse'].mean(), 2),
      '| mean ES RMSE:', round(fc['es_rmse'].mean(), 2))

## 2) Transportation Model (Lateral Transshipment)

Minimize `Z = ΣΣ c(i,j)·x(i,j)`  s.t. `Σ_j x(i,j) ≤ S(i)` (ต้นทาง), 
`Σ_i x(i,j) ≥ D(j)` (ปลายทาง), `x ≥ 0`  
โดย `c(i,j)` = ระยะทาง × transport_rate · `S(i)` = โดสเสี่ยงหมดอายุ (สถานะแดง ที่ขนส่งได้) · 
`D(j)` = ดีมานด์ที่ยังขาด

In [ ]:
from numpy import radians, sin, cos, arccos, clip
master = pd.read_csv(ROOT / 'data' / 'hospitals' / 'hospital_master.csv')
branches = pd.read_csv(VAX / 'vaccine_branches.csv')
vials = pd.read_csv(VAX / 'clean' / 'vaccine_vial_clean.csv')
vials['effective_expiry'] = pd.to_datetime(vials['effective_expiry'], utc=True, format='ISO8601')
BR = list(branches['hospital_id'])
loc = master.set_index('hospital_id').loc[BR, ['latitude', 'longitude']]
rate = branches.set_index('hospital_id')['transport_rate']

def haversine(a, b):
    la1, lo1, la2, lo2 = map(radians, [loc.loc[a, 'latitude'], loc.loc[a, 'longitude'],
                                       loc.loc[b, 'latitude'], loc.loc[b, 'longitude']])
    return 6371 * arccos(clip(sin(la1) * sin(la2) + cos(la1) * cos(la2) * cos(lo2 - lo1), -1, 1))

# เลือก 1 ผลิตภัณฑ์มาเดโม (mRNA) — supply = โดสในขวดเสี่ยง (<=14 วัน) ที่ขนส่งได้ (ไม่ใช่ OPENED)
now = pd.Timestamp('2026-06-26T08:00:00+07:00')
vials['days_remaining'] = (vials['effective_expiry'] - now).dt.total_seconds() / 86400
pid = 'VAX_MRNA_01'
risk = vials[(vials.product_id == pid) & (vials.state != 'OPENED') & (vials.days_remaining <= 14)]
supply = risk.groupby('hospital_id')['doses_remaining'].sum().reindex(BR, fill_value=0)
# demand ปลายทาง = ดีมานด์เฉลี่ย 7 วันล่าสุดของผลิตภัณฑ์นี้
recent = feat[(feat.product_id == pid) & (feat.date >= feat.date.max() - pd.Timedelta(days=7))]
demand_j = recent.groupby('hospital_id')['demand'].mean().reindex(BR, fill_value=0).round()
print('supply (เสี่ยงหมดอายุ):', supply.to_dict())
print('demand (เฉลี่ย 7วัน):  ', demand_j.to_dict())

In [ ]:
# สร้างเมทริกซ์ต้นทุน c(i,j) = ระยะทาง(กม.) * transport_rate(ปลายทาง)
n = len(BR)
C = np.zeros((n, n))
for i, si in enumerate(BR):
    for j, sj in enumerate(BR):
        C[i, j] = 0 if i == j else haversine(si, sj) * rate[sj]

# ตัวแปร x(i,j) แบนเป็นเวกเตอร์ — ลด ΣΣ c*x
c = C.flatten()
# ระบายของเสี่ยงทั้งหมดออกจากต้นทาง (equality): Σ_j x(i,j) = S(i)
A_eq, b_eq = [], []
for i in range(n):
    row = np.zeros(n * n); row[i * n:(i + 1) * n] = 1
    A_eq.append(row); b_eq.append(supply.iloc[i])
# ปลายทางรับได้ไม่เกินดีมานด์ (upper bound): Σ_i x(i,j) <= D(j)
A_ub, b_ub = [], []
for j in range(n):
    row = np.zeros(n * n)
    for i in range(n): row[i * n + j] = 1
    A_ub.append(row); b_ub.append(demand_j.iloc[j])
# ห้ามส่งหาตัวเอง (x(i,i)=0)
bounds = [(0, None) if i // n != i % n else (0, 0) for i in range(n * n)]
assert supply.sum() <= demand_j.sum(), 'supply เกิน demand รวม — ปรับ formulation'
res = linprog(c, A_ub=np.array(A_ub), b_ub=np.array(b_ub),
              A_eq=np.array(A_eq), b_eq=np.array(b_eq), bounds=bounds, method='highs')
print('LP status:', res.message, '| total cost Z =', round(res.fun or 0, 1))

In [ ]:
plan = pd.DataFrame(res.x.reshape(n, n), index=BR, columns=BR).round(1)
moved = plan.stack().reset_index()
moved.columns = ['from_hospital', 'to_hospital', 'doses']
moved = moved[moved['doses'] > 0.5].sort_values('doses', ascending=False)
moved['product_id'] = pid
moved.to_csv(OUT / 'transshipment_plan.csv', index=False, encoding='utf-8-sig')
print(f'แผนโอนย้าย {pid}: {len(moved)} เส้นทาง · รวม {moved.doses.sum():.0f} โดส')
moved